In [1]:
import pandas as pd
import numpy as np
from scipy.spatial import KDTree
 
def get_nearest_cells(builtup_df, target_cell, k=25):
    """
    En yakın hücreleri bulmak için KDTree kullanılır.
    """
    coords = builtup_df[['lat', 'lon']].values  # Hücre merkezlerinin koordinatları
    tree = KDTree(coords)  # KDTree oluştur
    distances, indices = tree.query(target_cell, k=k)  # En yakın k hücreyi bul
    return indices
 
def calculate_new_buildings(builtup_df, zoning_ratios, construction_areas, year, as_is_buildings):
    results = []
    for _, cell in builtup_df.iterrows():
        cell_id = cell['id']
        cell_area = cell['cell_area']
        saturation = cell[f'Saturation_updated_{year}']
        target_cell = (cell['lat'], cell['lon'])  # Hücre merkez koordinatları
 
        new_buildings = {'id': cell_id, 'left': cell['left'], 'top': cell['top'],
                         'right': cell['right'], 'bottom': cell['bottom'],'ilce': cell['ilce']}
        
        # Sadece saturasyon artışı olan hücrelerde işlem yap
        # Tüm yıllar için bir önceki yıl ile karşılaştırma yapıyoruz
        if saturation > cell[f'Saturation_updated_{year-1}']:
            # En yüksek imar oranını bulma
            max_zoning_type = None
            max_ratio = 0
    
            for zoning_type, ratio in zoning_ratios.loc[zoning_ratios['id'] == cell_id].iloc[:, 6:].items():
                ratio = float(ratio)
                if ratio > max_ratio:
                    max_ratio = ratio
                    max_zoning_type = zoning_type
    
            if max_zoning_type and max_ratio > 0.9:
                # En yüksek imar oranına sahip imar tipi için brüt alan kontrolü
                required_area = construction_areas.get(max_zoning_type, None)
                as_is_bina_sayisi = as_is_buildings.loc[as_is_buildings['id'] == cell_id, max_zoning_type].values[0]
                average_area = construction_areas.get(max_zoning_type, 1)
                as_is_area = as_is_bina_sayisi * average_area
    
                available_area = max(0, cell_area * saturation * max_ratio - as_is_area)
    
                if required_area and available_area < required_area:
                    # En yakın 25 hücrenin toplam alanını kontrol et
                    nearest_indices = get_nearest_cells(builtup_df, target_cell)
                    total_neighbor_area = sum(
                        max(0, builtup_df.iloc[idx]['cell_area'] * builtup_df.iloc[idx][f'Saturation_updated_{year}'] * max_ratio -
                            as_is_buildings.iloc[idx][max_zoning_type] * construction_areas.get(max_zoning_type, 1))
                        for idx in nearest_indices
                    )
                    if total_neighbor_area >= required_area:
                        new_buildings[max_zoning_type] = 1  # Alan yeterliyse bina sayısını 1 artır
                    else:
                        new_buildings[max_zoning_type] = 0  # Alan yetersizse 0
                else:
                    # Alan yeterliyse normal hesaplama yap
                    new_buildings[max_zoning_type] = int(np.floor(available_area / required_area)) if required_area else 0
            else:
                # Standart hesaplama: Diğer imar tipleri için
                for zoning_type, ratio in zoning_ratios.loc[zoning_ratios['id'] == cell_id].iloc[:, 6:].items():
                    ratio = float(ratio)
                    as_is_bina_sayisi = as_is_buildings.loc[as_is_buildings['id'] == cell_id, zoning_type].values[0]
                    average_area = construction_areas.get(zoning_type, 1)
                    as_is_area = as_is_bina_sayisi * average_area
                    available_area = max(0, cell_area * saturation * ratio - as_is_area)
    
                    if average_area is not None and average_area > 0:
                        new_buildings[zoning_type] = int(np.floor(available_area / average_area))
                    else:
                        new_buildings[zoning_type] = 0
        else:
            # Saturasyon artışı olmayan hücreler için tüm imar tiplerinde yeni bina 0 olarak ayarla
            for zoning_type in zoning_ratios.loc[zoning_ratios['id'] == cell_id].iloc[:, 6:].columns:
                new_buildings[zoning_type] = 0
 
        results.append(new_buildings)
 
    return pd.DataFrame(results)
 
# Girdi dosyalarını yükle
builtup_df = pd.read_excel(r'C:\Users\arda.sari\Desktop\SLF PROJE\SLF_hesabı\saturation_update_results_29-versiyon.xlsx')
construction_areas = pd.read_excel(r'C:\Users\arda.sari\Desktop\SLF PROJE\SLF_hesabı\construction_areas2.xlsx', sheet_name='areas')
 
# Ortalama bina alanlarını dictionary'e dönüştür
construction_areas_dict = construction_areas.set_index('imar_tipi')['bina_başı_brüt_alan'].to_dict()
 
# İlk yıl için as-is durumunu yükle
as_is_buildings = pd.read_excel(r"C:\Users\arda.sari\Desktop\SLF PROJE\SLF_hesabı\bina_sayisi_as-is.xlsx")
 
# Çıktı dosyasını oluştur
output_path = r'C:\Users\arda.sari\Desktop\SLF PROJE\SLF_hesabı\bina_sayisi_hesaplama_2024_2035_guncel_son_1703.xlsx'
new_buildings_output_path = r'C:\Users\arda.sari\Desktop\SLF PROJE\SLF_hesabı\new_buildings_2024_2035_1703.xlsx'
 
with pd.ExcelWriter(output_path, engine='openpyxl') as writer, pd.ExcelWriter(new_buildings_output_path, engine='openpyxl') as new_writer:
    for year in range(2024, 2036):
        print(f"Processing year {year}...")
 
        # Yıllık zoning ratios tablosunu yükle
        zoning_ratios = pd.read_excel(r'C:\Users\arda.sari\Desktop\SLF PROJE\SLF_hesabı\2024-2035_son_deneme_imar_tahmin_1203_4.xlsx', sheet_name=f'{year}')
 
        # Yeni bina sayıları hesapla
        new_buildings_df = calculate_new_buildings(builtup_df, zoning_ratios, construction_areas_dict, year, as_is_buildings)
        new_buildings_df.to_excel(new_writer, sheet_name=str(year), index=False)  # Yeni bina sayıları Excel'e yaz
 
        # Yeni bina sayıları ile as-is tablolarını birleştir
        as_is_buildings.set_index('id', inplace=True)
        new_buildings_df.set_index('id', inplace=True)
        combined_df = as_is_buildings.add(new_buildings_df, fill_value=0).reset_index()
 
        # Sütun sıralamasını düzenle (id, poligon koordinatları, imar tipleri)
        column_order = ['id', 'left', 'top', 'right', 'bottom', 'ilce'] + [col for col in combined_df.columns if col not in ['id', 'left', 'top', 'right', 'bottom','ilce']]
        combined_df = combined_df[column_order]
 
        # Çıktıyı Excel'e yaz
        combined_df.to_excel(writer, sheet_name=str(year), index=False)
 
        # Bir sonraki yıl için as-is durumunu güncelle
        as_is_buildings = combined_df.copy()
 
print("Sayılar başarıyla kaydedildi.")

Processing year 2024...


C:\Users\arda.sari\AppData\Local\Temp\ipykernel_8816\2806780757.py:33: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  ratio = float(ratio)
C:\Users\arda.sari\AppData\Local\Temp\ipykernel_8816\2806780757.py:65: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  ratio = float(ratio)
C:\Users\arda.sari\AppData\Local\Temp\ipykernel_8816\2806780757.py:33: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  ratio = float(ratio)
C:\Users\arda.sari\AppData\Local\Temp\ipykernel_8816\2806780757.py:65: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  ratio = float(ratio)
C:\Users\arda.sari\AppData\Local\Temp\ipykernel_8816

Processing year 2025...


C:\Users\arda.sari\AppData\Local\Temp\ipykernel_8816\2806780757.py:33: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  ratio = float(ratio)
C:\Users\arda.sari\AppData\Local\Temp\ipykernel_8816\2806780757.py:65: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  ratio = float(ratio)
C:\Users\arda.sari\AppData\Local\Temp\ipykernel_8816\2806780757.py:33: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  ratio = float(ratio)
C:\Users\arda.sari\AppData\Local\Temp\ipykernel_8816\2806780757.py:65: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  ratio = float(ratio)
C:\Users\arda.sari\AppData\Local\Temp\ipykernel_8816

Processing year 2026...


...

Processing year 2034...


C:\Users\arda.sari\AppData\Local\Temp\ipykernel_8816\2806780757.py:33: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  ratio = float(ratio)
C:\Users\arda.sari\AppData\Local\Temp\ipykernel_8816\2806780757.py:65: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  ratio = float(ratio)
C:\Users\arda.sari\AppData\Local\Temp\ipykernel_8816\2806780757.py:33: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  ratio = float(ratio)
C:\Users\arda.sari\AppData\Local\Temp\ipykernel_8816\2806780757.py:65: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  ratio = float(ratio)
C:\Users\arda.sari\AppData\Local\Temp\ipykernel_8816

Processing year 2035...


C:\Users\arda.sari\AppData\Local\Temp\ipykernel_8816\2806780757.py:33: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  ratio = float(ratio)
C:\Users\arda.sari\AppData\Local\Temp\ipykernel_8816\2806780757.py:65: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  ratio = float(ratio)
C:\Users\arda.sari\AppData\Local\Temp\ipykernel_8816\2806780757.py:33: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  ratio = float(ratio)
C:\Users\arda.sari\AppData\Local\Temp\ipykernel_8816\2806780757.py:65: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  ratio = float(ratio)
C:\Users\arda.sari\AppData\Local\Temp\ipykernel_8816

Sayılar başarıyla kaydedildi.
